# 📘 Notebook – Tabela 1

**Acesso aos exames preventivos de saúde (PNS 2013 e 2019)**
Mulheres com 25 anos ou mais

In [3]:
# Importação da camada de serviço (abstração semântica da PNS)

import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Setup do ambiente
sys.path.append(str(Path("..").resolve()))
from service.pns_service import (
    get_dataframe,
    list_variables,
    register_derived_variable
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 🔹 Célula 2 – Inspeção das variáveis disponíveis (bronze)

Objetivo: confirmar **empiricamente** quais variáveis existem para exames preventivos.

In [4]:
df_vars = list_variables()

df_vars[df_vars["variable"].str.contains(
    "preventivo|mamografia", case=False, na=False
)]

,variable,descricao,categoria,tipo_dado,regra_derivacao,depends_on,code_2013,type_2013,exists_2013,code_2019,type_2019,exists_2019
4,fez_mamografia,Flag: realizou mamografia,derivada,float,Calculada a partir de: mamografia_quando,[mamografia_quando],None,None,False,None,None,False
5,fez_preventivo,Flag: realizou exame preventivo (Papanicolau) ...,derivada,float,Calculada a partir de: preventivo_quando,[preventivo_quando],None,None,False,None,None,False
10,mamografia_cobertura,Tipo de cobertura do exame de mamografia. Cate...,derivada,float,"Calculada a partir de: mamografia_sus, mamogra...","[mamografia_sus, mamografia_plano, mamografia_...",None,None,False,None,None,False
11,mamografia_fez,Indicador (0/1) se a mulher realizou exame de ...,derivada,float,Calculada a partir de: mamografia_quando,[mamografia_quando],None,None,False,None,None,False
12,mamografia_paga,A Sra pagou algum valor pela última mamografia...,fisica,string,None,None,R019,string,True,R019,string,True
13,mamografia_plano,A última mamografia foi coberta por plano de s...,fisica,string,None,None,R018,string,True,None,string,False
14,mamografia_quando,\n Quando foi a última vez que a Sra fe...,fisica,string,None,None,R017,string,True,R01701,string,True
15,mamografia_sus,A última mamografia foi feita pelo SUS? (1=Sim...,fisica,string,None,None,R020,string,True,R020,string,True
17,preventivo_cobertura,Tipo de cobertura do exame preventivo (papanic...,derivada,float,"Calculada a partir de: preventivo_sus, prevent...","[preventivo_sus, preventivo_plano, preventivo_...",None,None,False,None,None,False
18,preventivo_fez,Indicador (0/1) se a mulher realizou exame pre...,derivada,float,Calculada a partir de: preventivo_quando,[preventivo_quando],None,None,False,None,None,False


## 🔹 Célula 3 – Carregamento mínimo para inspeção empírica

Objetivo: **ver os valores reais**, sem ainda criar nenhuma variável analítica.

In [5]:
variables = [
    "preventivo_quando",
    "mamografia_quando"
]

sources = ["2013", "2019"]

df_raw = get_dataframe(
    variables=variables,
    sources=sources
)

df_raw.head()

,preventivo_quando,id_domicilio,id_upa,mamografia_quando,origem,id_morador
0,2,0001,1100001,None,2013,1
1,5,0002,1100001,None,2013,1
2,3,0003,1100001,None,2013,2
3,2,0009,1100001,None,2013,1
4,None,0010,1100001,None,2013,2


## 🔹 Célula 4 – Conferência estrutural e de domínio (obrigatória)

In [6]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158912 entries, 0 to 158911
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   preventivo_quando  72514 non-null   object
 1   id_domicilio       158912 non-null  object
 2   id_upa             158912 non-null  object
 3   mamografia_quando  33476 non-null   object
 4   origem             158912 non-null  object
 5   id_morador         158912 non-null  object
dtypes: object(6)
memory usage: 7.3+ MB


In [7]:
for col in variables:
    print(f"\nDistribuição de valores em '{col}':")
    print(df_raw[col].value_counts(dropna=False))


Distribuição de valores em 'preventivo_quando':
preventivo_quando
None    86398
1       30539
2       16630
4       12121
5        6799
3        6425
Name: count, dtype: int64

Distribuição de valores em 'mamografia_quando':
mamografia_quando
None    125436
1        17325
2         7970
4         5151
3         3030
Name: count, dtype: int64


📌 **Conclusão empírica (documentada):**

* Valores `"1"`, `"2"`, `"3"`, `"4"` indicam que o exame foi realizado;
* Qualquer outro valor (`None`, `"5"`, etc.) indica que **não foi realizado**;
* A regra é **simétrica e válida para ambos os exames**.

## 🔹 Célula 5 – Registro das variáveis GOLD (persistentes)

Aqui formalizamos a regra analítica como **variáveis derivadas persistidas** no SQLite.

In [8]:
RANGE_FEZ = {"1", "2", "3", "4"}

register_derived_variable(
    name="preventivo_fez",
    description=(
        "Indicador (0/1) se a mulher realizou exame preventivo "
        "(papanicolau). Assume 1 quando preventivo_quando ∈ {1,2,3,4}, "
        "e 0 caso contrário."
    ),
    depends_on=["preventivo_quando"],
    func=lambda df: df["preventivo_quando"].isin(RANGE_FEZ).astype(int)
)

register_derived_variable(
    name="mamografia_fez",
    description=(
        "Indicador (0/1) se a mulher realizou exame de mamografia. "
        "Assume 1 quando mamografia_quando ∈ {1,2,3,4}, "
        "e 0 caso contrário."
    ),
    depends_on=["mamografia_quando"],
    func=lambda df: df["mamografia_quando"].isin(RANGE_FEZ).astype(int)
)

📌 Essas variáveis passam a compor o **nível gold**:

* documentadas no catálogo;
* persistidas no banco;
* reutilizáveis em qualquer notebook futuro.

## 🔹 Célula 6 – Forçar cálculo e persistência no SQLite

In [9]:
# Verificação de sanidade: distribuição das variáveis derivadas

df_gold = get_dataframe(
    variables=["preventivo_fez", "mamografia_fez"],
    sources=["2013", "2019"]
)

df_gold.head()

,id_domicilio,preventivo_fez,id_upa,origem,id_morador,mamografia_fez
0,0001,1,1100001,2013,1,0
1,0002,0,1100001,2013,1,0
2,0003,1,1100001,2013,2,0
3,0009,1,1100001,2013,1,0
4,0010,0,1100001,2013,2,0


## 🔹 Célula 7 – Validação empírica das variáveis gold

In [10]:
print("preventivo_fez:")
print(df_gold["preventivo_fez"].value_counts(dropna=False))

print("\nmamografia_fez:")
print(df_gold["mamografia_fez"].value_counts(dropna=False))

preventivo_fez:
preventivo_fez
0    93197
1    65715
Name: count, dtype: int64

mamografia_fez:
mamografia_fez
0    125436
1     33476
Name: count, dtype: int64


## 🔹 Célula 8 – Função auxiliar para cálculo da Tabela 1

In [11]:
def tabela_acesso(df, col):
    """
    Calcula a proporção (%) de mulheres que realizaram o exame,
    retornando uma série com Fez, Não fez e Total.
    """
    # Converter explicitamente para numérico
    x = pd.to_numeric(df[col], errors="coerce")

    fez = (x.mean() * 100).round(2)
    nao_fez = (100 - fez).round(2)

    return pd.Series(
        [fez, nao_fez, 100.0],
        index=["Fez", "Não fez", "Total"]
    )


## 🔹 Célula 9 – Construção da Tabela 1 (por ano)

In [12]:
tabelas = {}

for ano in sorted(df_gold["origem"].unique()):
    df_ano = df_gold[df_gold["origem"] == ano]
    
    tabela = pd.DataFrame({
        "Papanicolau": tabela_acesso(df_ano, "preventivo_fez"),
        "Mamografia": tabela_acesso(df_ano, "mamografia_fez")
    })
    
    tabelas[ano] = tabela

## 🔹 Célula 10 – Exibição final (formato artigo)

In [16]:
for ano, tabela in tabelas.items():
    print(f"\nTabela 1 – Acesso aos serviços preventivos (%), PNS {ano}")
    display(tabela)






Tabela 1 – Acesso aos serviços preventivos (%), PNS 2013


,Papanicolau,Mamografia
Fez,41.33,19.49
Não fez,58.67,80.51
Total,100.00,100.00



Tabela 1 – Acesso aos serviços preventivos (%), PNS 2019


,Papanicolau,Mamografia
Fez,41.37,22.14
Não fez,58.63,77.86
Total,100.00,100.00


In [20]:
for nome, df in tabelas.items():
    print(f"Tabela 1 — PNS {nome}")
    display(df)





Tabela 1 — PNS 2013


,Papanicolau,Mamografia
Fez,41.33,19.49
Não fez,58.67,80.51
Total,100.00,100.00


Tabela 1 — PNS 2019


,Papanicolau,Mamografia
Fez,41.37,22.14
Não fez,58.63,77.86
Total,100.00,100.00
